# Metabolite Availability - CELL MESH Integration Test

这是 cellmesh 包的 metabolite availability 模块集成测试 notebook。

注意：本 notebook 所有生产计算都调用 cellmesh 包内真实方法，不在 notebook 中重新实现算法。

In [1]:
# 1. 导入 cellmesh
import cellmesh
import numpy as np
import pandas as pd
from scipy import sparse
from IPython.display import display

print(f"cellmesh 导入路径: {cellmesh.__file__}")
print(f"cellmesh 版本: {cellmesh.__version__}")
print(f"✓ compute_metabolite_availability 可用: {'compute_metabolite_availability' in dir(cellmesh)}")

cellmesh 导入路径: /home/qsong/.openclaw/workspace/developer/cell_mesh_pkg/cellmesh/__init__.py
cellmesh 版本: 0.3.0
✓ compute_metabolite_availability 可用: True


## 2. 生成 Toy AnnData

按照文档要求，使用指定的基因和细胞类型。

In [2]:
# 2.1 定义基因和细胞类型
genes = [
    "G_prodA1", "G_prodA2", "G_prodA3",
    "G_consA",
    "G_transA1", "G_transA2",
    "G_prodB", "G_prodF",
    "G_noise"
]

cells = ["T1", "T2", "T3", "M1", "M2", "M3", "F1", "F2"]

cell_types = [
    "T_cell", "T_cell", "T_cell",
    "Macrophage", "Macrophage", "Macrophage",
    "Fibroblast", "Fibroblast"
]

# 2.2 构建表达矩阵（已经是 log-normalized）
X = np.array([
    [4.0, 1.0, 0.7, 0.2, 2.5, 1.5, 0.1, 1.2, 0.5],
    [0, 1.2, 0.8, 0.3, 2.7, 1.4, 0.0, 1.1, 0.4],
    [0, 0.8, 0.6, 0.2, 2.4, 1.6, 0.2, 1.3, 0.6],
    [1.0, 2.8, 0.5, 4.0, 0.8, 0.4, 1.0, 0.5, 0.7],
    [1.2, 3.0, 0.4, 4.2, 0.7, 0.3, 0.8, 0.6, 0.8],
    [0.9, 2.6, 0.6, 3.8, 0.9, 0.5, 1.1, 0.4, 0.6],
    [0.5, 0.6, 0.2, 0.8, 0.3, 0.2, 4.0, 2.5, 0.9],
    [0.4, 0.7, 0.3, 0.9, 0.4, 0.2, 4.2, 2.7, 1.0],
], dtype=float)

In [3]:
# 2.3 生成简单的 AnnData（或者用真实的 anndata）
class SimpleAnnData:
    def __init__(self, X, var_names, obs):
        self.X = X
        self.layers = {}
        self.var_names = pd.Index(var_names)
        self.obs = pd.DataFrame(obs)

adata = SimpleAnnData(
    X=X,
    var_names=genes,
    obs={"cell_type": cell_types}
)

# 2.4 展示 cell × gene 表达矩阵
print("AnnData shape:", adata.X.shape)
expr_df = pd.DataFrame(adata.X, index=cells, columns=genes)
expr_df.insert(0, "cell_type", cell_types)
display(expr_df.round(3))

AnnData shape: (8, 9)


,cell_type,G_prodA1,G_prodA2,G_prodA3,G_consA,G_transA1,G_transA2,G_prodB,G_prodF,G_noise
T1,T_cell,4.0,1.0,0.7,0.2,2.5,1.5,0.1,1.2,0.5
T2,T_cell,0.0,1.2,0.8,0.3,2.7,1.4,0.0,1.1,0.4
T3,T_cell,0.0,0.8,0.6,0.2,2.4,1.6,0.2,1.3,0.6
M1,Macrophage,1.0,2.8,0.5,4.0,0.8,0.4,1.0,0.5,0.7
M2,Macrophage,1.2,3.0,0.4,4.2,0.7,0.3,0.8,0.6,0.8
M3,Macrophage,0.9,2.6,0.6,3.8,0.9,0.5,1.1,0.4,0.6
F1,Fibroblast,0.5,0.6,0.2,0.8,0.3,0.2,4.0,2.5,0.9
F2,Fibroblast,0.4,0.7,0.3,0.9,0.4,0.2,4.2,2.7,1.0


## 3. 调用 cellmesh 构建 Pseudobulk

通过 compute_metabolite_availability 的中间结果来获取 pseudobulk。

In [4]:
# 3.1 先生成一个简单的 reaction table 来调用 API
simple_reaction = pd.DataFrame([
    {"metabolite": "Test", "HMDB_ID": "HMDB000", "reaction": "R0", "gene": "G_noise", "direction": "product"}
])

# 3.2 调用 compute_metabolite_availability 获取 pseudobulk
simple_result = cellmesh.compute_metabolite_availability(
    adata,
    simple_reaction,
    celltype_col="cell_type",
    min_cells=1,
    lower=0,
    upper=100,
    return_intermediates=True
)

# 3.3 展示 pseudobulk
print("\nCell type pseudobulk / cell_type × gene:")
pseudobulk = simple_result['pseudobulk']
display(pseudobulk.round(3))


Cell type pseudobulk / cell_type × gene:


,G_prodA1,G_prodA2,G_prodA3,G_consA,G_transA1,G_transA2,G_prodB,G_prodF,G_noise
T_cell,1.333,1.00,0.70,0.233,2.533,1.5,0.100,1.2,0.50
Macrophage,1.033,2.80,0.50,4.000,0.800,0.4,0.967,0.5,0.70
Fibroblast,0.450,0.65,0.25,0.850,0.350,0.2,4.100,2.6,0.95


In [5]:
simple_result.keys()

dict_keys(['availability', 'metadata', 'P', 'C', 'E', 'P_norm', 'C_norm', 'E_norm', 'pseudobulk', 'reaction_genes'])

In [6]:
simple_result['metadata']

,,has_product,has_substrate,has_transporter,n_product_reactions,n_substrate_reactions,n_transporter_reactions
metabolite,hmdb_id,,,,,,
Test,HMDB000,True,False,False,1,0,0


## 4. 构建 Reaction Table

按照文档要求构建完整的 reaction table。

In [7]:
# 4.1 构建完整的 reaction table
reaction_table = pd.DataFrame([
    #{"metabolite": "MetA", "HMDB_ID": "HMDB00001", "reaction": "R_A_prod_1", "gene": "G_prodA1", "direction": "product"},
    {"metabolite": "MetA", "HMDB_ID": "HMDB00001", "reaction": "R_A_prod_1", "gene": "G_prodA2, G_prodA1", "direction": "product"},
    {"metabolite": "MetA", "HMDB_ID": "HMDB00001", "reaction": "R_A_prod_2", "gene": "G_prodA3", "direction": "product"},
    {"metabolite": "MetA", "HMDB_ID": "HMDB00001", "reaction": "R_A_sub_1", "gene": "G_consA", "direction": "substrate"},
    {"metabolite": "MetA", "HMDB_ID": "HMDB00001", "reaction": "R_A_trans_1", "gene": "G_transA1", "direction": "transporter"},
    {"metabolite": "MetA", "HMDB_ID": "HMDB00001", "reaction": "R_A_trans_1", "gene": "G_transA2", "direction": "transporter"},
    {"metabolite": "MetB", "HMDB_ID": "HMDB00002", "reaction": "R_B_prod_1", "gene": "G_prodB", "direction": "product"},
    {"metabolite": "MetF", "HMDB_ID": "HMDB00006", "reaction": "R_F_prod_1", "gene": "G_prodF", "direction": "product"},
    {"metabolite": "MetF", "HMDB_ID": "HMDB00006", "reaction": "R_F_sub_1", "gene": "G_consA", "direction": "substrate"},
    {"metabolite": "MetD", "HMDB_ID": "HMDB00004", "reaction": "R_D_sub_1", "gene": "G_consA", "direction": "substrate"},
    {"metabolite": "MetC", "HMDB_ID": "HMDB00003", "reaction": "R_C_prod_1", "gene": "G_missing", "direction": "product"},
])

print("Reaction Table:")
display(reaction_table)

Reaction Table:


,metabolite,HMDB_ID,reaction,gene,direction
0,MetA,HMDB00001,R_A_prod_1,"G_prodA2, G_prodA1",product
1,MetA,HMDB00001,R_A_prod_2,G_prodA3,product
2,MetA,HMDB00001,R_A_sub_1,G_consA,substrate
3,MetA,HMDB00001,R_A_trans_1,G_transA1,transporter
4,MetA,HMDB00001,R_A_trans_1,G_transA2,transporter
5,MetB,HMDB00002,R_B_prod_1,G_prodB,product
6,MetF,HMDB00006,R_F_prod_1,G_prodF,product
7,MetF,HMDB00006,R_F_sub_1,G_consA,substrate
8,MetD,HMDB00004,R_D_sub_1,G_consA,substrate
9,MetC,HMDB00003,R_C_prod_1,G_missing,product


## 5. 调用 cellmesh.compute_metabolite_availability

In [8]:
# 5.1 调用 API
result = cellmesh.compute_metabolite_availability(
    adata,
    reaction_table,
    celltype_col="cell_type",
    min_cells=1,
    lower=0,
    upper=100,
    eps=0.05,
    beta=0.5,
    missing_C_norm=0.41,
    missing_E_norm=0.75,
    return_intermediates=True
)

In [9]:
result.keys()

dict_keys(['availability', 'metadata', 'P', 'C', 'E', 'P_norm', 'C_norm', 'E_norm', 'pseudobulk', 'reaction_genes'])

In [10]:
result['metadata']

,,has_product,has_substrate,has_transporter,n_product_reactions,n_substrate_reactions,n_transporter_reactions
metabolite,hmdb_id,,,,,,
MetF,HMDB00006,True,True,False,1,1,0
MetA,HMDB00001,True,True,True,2,1,1
MetB,HMDB00002,True,False,False,1,0,0


In [11]:
print("\nP (Product scores):")
display(result['P'].round(3))


P (Product scores):


,,T_cell,Macrophage,Fibroblast
metabolite,hmdb_id,,,
MetF,HMDB00006,1.200,0.500,2.6
MetA,HMDB00001,2.033,3.300,0.9
MetB,HMDB00002,0.100,0.967,4.1


## 6. 展示中间结果

In [12]:
print("\nP (Product scores):")
display(result['P'].round(3))

print("\nC (Substrate scores):")
display(result['C'].round(3))

print("\nE (Transporter scores):")
display(result['E'].round(3))


P (Product scores):


,,T_cell,Macrophage,Fibroblast
metabolite,hmdb_id,,,
MetF,HMDB00006,1.200,0.500,2.6
MetA,HMDB00001,2.033,3.300,0.9
MetB,HMDB00002,0.100,0.967,4.1



C (Substrate scores):


,,T_cell,Macrophage,Fibroblast
metabolite,hmdb_id,,,
MetF,HMDB00006,0.233,4.0,0.85
MetA,HMDB00001,0.233,4.0,0.85
MetB,HMDB00002,0.000,0.0,0.00



E (Transporter scores):


,,T_cell,Macrophage,Fibroblast
metabolite,hmdb_id,,,
MetF,HMDB00006,0.000,0.0,0.00
MetA,HMDB00001,2.533,0.8,0.35
MetB,HMDB00002,0.000,0.0,0.00


In [13]:
print("\nP_norm (Normalized product scores):")
display(result['P_norm'].round(3))

print("\nC_norm (Normalized substrate scores):")
display(result['C_norm'].round(3))

print("\nE_norm (Normalized transporter scores):")
display(result['E_norm'].round(3))

print("\nMetadata:")
display(result['metadata'])


P_norm (Normalized product scores):


,,T_cell,Macrophage,Fibroblast
metabolite,hmdb_id,,,
MetF,HMDB00006,0.333,0.000,1.0
MetA,HMDB00001,0.472,1.000,0.0
MetB,HMDB00002,0.000,0.217,1.0



C_norm (Normalized substrate scores):


,,T_cell,Macrophage,Fibroblast
metabolite,hmdb_id,,,
MetF,HMDB00006,0.00,1.00,0.164
MetA,HMDB00001,0.00,1.00,0.164
MetB,HMDB00002,0.41,0.41,0.410



E_norm (Normalized transporter scores):


,,T_cell,Macrophage,Fibroblast
metabolite,hmdb_id,,,
MetF,HMDB00006,0.75,0.750,0.75
MetA,HMDB00001,1.00,0.206,0.00
MetB,HMDB00002,0.75,0.750,0.75



Metadata:


,,has_product,has_substrate,has_transporter,n_product_reactions,n_substrate_reactions,n_transporter_reactions
metabolite,hmdb_id,,,,,,
MetF,HMDB00006,True,True,False,1,1,0
MetA,HMDB00001,True,True,True,2,1,1
MetB,HMDB00002,True,False,False,1,0,0


## 7. 展示 Metabolite Availability

In [14]:
print("\nMetabolite Availability (metabolite × cell_type):")
availability = result['availability']
display(availability.round(4))


Metabolite Availability (metabolite × cell_type):


,,T_cell,Macrophage,Fibroblast
metabolite,hmdb_id,,,
MetF,HMDB00006,0.3142,0.0089,0.7908
MetA,HMDB00001,0.5619,0.0601,0.0024
MetB,HMDB00002,0.0320,0.1707,0.6720


## 8. Dense vs Sparse 一致性测试

In [15]:
# 8.1 创建 sparse AnnData
adata_sparse = SimpleAnnData(
    X=sparse.csr_matrix(X),
    var_names=genes,
    obs={"cell_type": cell_types}
)

# 8.2 用 sparse 输入计算
result_sparse = cellmesh.compute_metabolite_availability(
    adata_sparse,
    reaction_table,
    celltype_col="cell_type",
    min_cells=1,
    lower=0,
    upper=100,
    eps=0.05,
    beta=0.5,
    missing_C_norm=0.41,
    missing_E_norm=0.75,
    return_intermediates=True
)

# 8.3 比较结果
print("\nDense vs Sparse availability difference:")
diff = availability - result_sparse['availability']
display(diff.round(10))

# 8.4 断言一致性
pd.testing.assert_frame_equal(
    availability,
    result_sparse['availability'],
    atol=1e-10
)
print("\n✓ Dense vs Sparse 输入结果一致！")


Dense vs Sparse availability difference:


,,T_cell,Macrophage,Fibroblast
metabolite,hmdb_id,,,
MetF,HMDB00006,0.0,0.0,0.0
MetA,HMDB00001,0.0,0.0,0.0
MetB,HMDB00002,0.0,0.0,0.0



✓ Dense vs Sparse 输入结果一致！


## 9. 断言验证 Toy Data

In [16]:
print("\n" + "=" * 70)
print("Running Toy Assertion Tests")
print("=" * 70)

# 找到 MetA 的索引
metA_idx = None
for idx in result['P'].index:
    if idx[0] == 'MetA':
        metA_idx = idx
        break

print(f"\nTesting MetA at index: {metA_idx}")

# 1. 测试多基因 reaction 取 max 和多个 reaction 求和
expected_metA_P = pseudobulk[["G_prodA1", "G_prodA2"]].max(axis=1) + pseudobulk["G_prodA3"]
actual_metA_P = result['P'].loc[metA_idx]
print(f"\nExpected P: {expected_metA_P.to_dict()}")
print(f"Actual P: {actual_metA_P.to_dict()}")
pd.testing.assert_series_equal(expected_metA_P, actual_metA_P, check_names=False, atol=1e-10)
print("✓ Multi-gene takes max; multi-reaction sums")

# 2. 测试 MetB 的默认值
metB_idx = None
for idx in result['C_norm'].index:
    if idx[0] == 'MetB':
        metB_idx = idx
        break

if metB_idx is not None:
    assert np.allclose(result['C_norm'].loc[metB_idx].values, 0.41)
    assert np.allclose(result['E_norm'].loc[metB_idx].values, 0.75)
    print("✓ MetB uses defaults C_norm=0.41 and E_norm=0.75")

# 3. 测试 MetF 的默认 E_norm
metF_idx = None
for idx in result['E_norm'].index:
    if idx[0] == 'MetF':
        metF_idx = idx
        break

if metF_idx is not None:
    assert np.allclose(result['E_norm'].loc[metF_idx].values, 0.75)
    print("✓ MetF uses default E_norm=0.75")

# 4. 验证 MetD 和 MetC 被跳过
included_mets = [idx[0] for idx in availability.index]
print(f"\nIncluded metabolites: {included_mets}")
assert 'MetD' not in included_mets, 'MetD should be skipped'
assert 'MetC' not in included_mets, 'MetC should be skipped'
print("✓ MetD (no product) and MetC (missing genes) are skipped")

# 5. 验证 availability 形状正确
print(f"\nAvailability shape: {availability.shape}")
assert availability.shape[1] == 3, 'Should have 3 cell types'
print("✓ Availability shape is correct")

print("\n" + "=" * 70)
print("✓ All Toy Assertion Tests Passed!")
print("=" * 70)


Running Toy Assertion Tests

Testing MetA at index: ('MetA', 'HMDB00001')

Expected P: {'T_cell': 2.033333333333333, 'Macrophage': 3.3000000000000003, 'Fibroblast': 0.8999999999999999}
Actual P: {'T_cell': 2.033333333333333, 'Macrophage': 3.3000000000000003, 'Fibroblast': 0.8999999999999999}
✓ Multi-gene takes max; multi-reaction sums
✓ MetB uses defaults C_norm=0.41 and E_norm=0.75
✓ MetF uses default E_norm=0.75

Included metabolites: ['MetF', 'MetA', 'MetB']
✓ MetD (no product) and MetC (missing genes) are skipped

Availability shape: (3, 3)
✓ Availability shape is correct

✓ All Toy Assertion Tests Passed!


## 10. 与下游 CELL MESH 通讯流程兼容性测试

In [17]:
print("\n" + "=" * 70)
print("Testing Downstream Compatibility with CELL MESH")
print("=" * 70)

# 加载数据库
enzyme, sensor = cellmesh.load_cell_mesh_database()
print(f"\n✓ Loaded enzyme table: {len(enzyme)} rows")
print(f"✓ Loaded sensor table: {len(sensor)} rows")

# 用新方法运行 CELL MESH
print("\nRunning CELL MESH with use_new_availability=True...")
try:
    res = cellmesh.run_cell_mesh(
        adata,
        reaction_table=reaction_table,
        use_new_availability=True,
        cell_type_key="cell_type",
        min_cells_per_group=1,
        allow_self=False,
        lower=0,
        upper=100
    )
    print(f"✓ CELL MESH completed with {len(res.events)} events!")
    if not res.events.empty:
        print(f"\nTop events:\n{res.events[['sender', 'receiver', 'metabolite', 'cell_mesh_score']].head()}")
    print("\n✓ Downstream cell-cell communication is compatible!")
except Exception as e:
    print(f"⚠️ CELL MESH test: {e}")
    print("This may be normal if no matching sensor/enzyme pairs in test data")

print("\n" + "=" * 70)
print("✓ Integration Test Complete!")
print("=" * 70)


Testing Downstream Compatibility with CELL MESH

✓ Loaded enzyme table: 109 rows
✓ Loaded sensor table: 30 rows

Running CELL MESH with use_new_availability=True...
⚠️ CELL MESH test: No enzyme prior genes found in adata.var_names
This may be normal if no matching sensor/enzyme pairs in test data

✓ Integration Test Complete!


In [18]:
enzyme

,metabolite,hmdb_id,gene,role,weight,evidence_level,source,reaction
0,12-oxo-Leukotriene B4,HMDB0004234,PTGR2,production,1.0,Enzyme,packaged_enzyme_test,Leukotriene B4 + NADP+ → 12-oxo-Leukotriene B4...
1,12-oxo-Leukotriene B4,HMDB0004234,PTGR1,production,1.0,Enzyme,packaged_enzyme_test,Leukotriene B4 + NADP+ → 12-oxo-Leukotriene B4...
2,12-oxo-Leukotriene B4,HMDB0004234,PTGR3,production,1.0,Enzyme,packaged_enzyme_test,Leukotriene B4 + NADP+ → 12-oxo-Leukotriene B4...
3,12-oxo-Leukotriene B4,HMDB0004234,PTGR2,production,1.0,Enzyme,packaged_enzyme_test,Leukotriene B4 + NAD+ → 12-oxo-Leukotriene B4 ...
4,12-oxo-Leukotriene B4,HMDB0004234,PTGR1,production,1.0,Enzyme,packaged_enzyme_test,Leukotriene B4 + NAD+ → 12-oxo-Leukotriene B4 ...
...,...,...,...,...,...,...,...,...
104,Cholesterol,HMDB0000067,SULT2B1,degradation,1.0,Enzyme,packaged_enzyme_test,Cholesterol + Phosphoadenosine phosphosulfate ...
105,Cholesterol,HMDB0000067,CH25H,degradation,1.0,Unknown,packaged_enzyme_test,Cholesterol + Reduced acceptor + Oxygen → 25-H...
106,Cholesterol,HMDB0000067,CYP11A1,degradation,1.0,Unknown,packaged_enzyme_test,Cholesterol + reduced adrenal ferredoxin + Oxy...
107,Cholesterol,HMDB0000067,LIPA,production,1.0,Enzyme,packaged_enzyme_test,Cholesterol ester + Water → Cholesterol + Fatt...


In [19]:
sensor 

,metabolite,hmdb_id,sensor_gene,sensor_type,weight,evidence_level,source,protein_name,reference
0,12-oxo-Leukotriene B4,HMDB0004234,LTB4R,surface_receptor,1.0,Cell surface receptor,"[CellphoneDB, MetalinksDB, MEBOCOST]",LT4R1_HUMAN,NaN
1,12-oxo-Leukotriene B4,HMDB0004234,LTB4R2,surface_receptor,1.0,Cell surface receptor,"[CellphoneDB, MetalinksDB, MEBOCOST]",LT4R2_HUMAN,NaN
2,2-arachidonoylglycerol,HMDB0004666,CNR1,transporter,1.0,Transporter,"[CellphoneDB, MetalinksDB, MEBOCOST]",CNR1_HUMAN,NaN
3,2-arachidonoylglycerol,HMDB0004666,CNR2,surface_receptor,1.0,Other receptor,"[CellphoneDB, MetalinksDB, scConnect, MEBOCOST]",CNR2_HUMAN,NaN
4,2-arachidonoylglycerol,HMDB0004666,GPR55,surface_receptor,1.0,Cell surface receptor,"[CellphoneDB, MetalinksDB, scConnect, MEBOCOST]",GPR55_HUMAN,NaN
5,22-Hydroxycholesterol,HMDB0004035,NR1H4,nuclear_receptor,1.0,Nuclear receptor,"[CellphoneDB, MetalinksDB, scConnect]",NR1H4_HUMAN,NaN
6,5-α-Dihydroprogesterone,HMDB0003759,PGR,surface_receptor,1.0,Other receptor,"[CellphoneDB, MetalinksDB, MEBOCOST]",PRGR_HUMAN,NaN
7,5-S-HpETE,HMDB0001193,OXER1,surface_receptor,1.0,Cell surface receptor,"[CellphoneDB, MetalinksDB, MRCLinkdb, scConnec...",OXER1_HUMAN,NaN
8,Acetylcholine,HMDB0000895,CHRNA2,surface_receptor,1.0,Cell surface receptor,"[CellphoneDB, MEBOCOST, MetalinksDB, MRCLinkdb...",ACHA2_HUMAN,NaN
9,Acetylcholine,HMDB0000895,CHRNB3,surface_receptor,1.0,Cell surface receptor,"[CellphoneDB, MetalinksDB, MRCLinkdb, NeuronCh...",ACHB3_HUMAN,NaN


In [20]:
adata.var_names

Index(['G_prodA1', 'G_prodA2', 'G_prodA3', 'G_consA', 'G_transA1', 'G_transA2',
       'G_prodB', 'G_prodF', 'G_noise'],
      dtype='object')

In [21]:
adata = cellmesh.read_anndata('../../demo_HNSC_200cell.h5ad', mode='h5ad')

In [23]:
adata.obs

,UMAP_1,UMAP_2,Celltype (malignancy),celltype,Celltype (minor-lineage),Celltype (original),cluster,Site,Celltype,Patient,Source,Age,Gender,Stage,TNMstage
Cell,,,,,,,,,,,,,,,
HNSCC5_p14_HNSCC5_P14_LN_H02,-7.213791,7.989740,Stromal cells,Fibroblasts,Fibroblasts,Fibroblast,5,Lymph node,Fibroblast,HNSCC5,tLN,69,Female,Primary,II
HNSCC_17_P10_C11_S227_comb,11.418259,1.437914,Immune cells,CD8T,CD8Tcm,T cell,6,Primary,T cell,HNSCC9,Tumor,77,Female,Primary,II
HNSCC6_p15_HNSCC6_P15_LN_D05,-3.294716,-11.959078,Malignant cells,Malignant,Malignant,Malignant,12,Primary,Malignant,HNSCC6,Tumor,88,Female,Primary,I
HNSCC25_P2_A10_S10_comb,-7.387272,4.525463,Stromal cells,Myofibroblasts,Myofibroblasts,Fibroblast,2,Primary,-Fibroblast,HNSCC25,Tumor,76,Female,Primary,II
HNSCC16_P4_HNSCC16_P4_B11,-16.571148,6.022232,Stromal cells,Myocyte,Myocyte,myocyte,20,Primary,myocyte,HNSCC16,Tumor,63,Female,Primary,I
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
HNSCC18_P2_F02_S158_comb,-3.270715,-5.763773,Malignant cells,Malignant,Malignant,Malignant,8,Primary,Malignant,HNSCC18,Tumor,41,Male,Primary,II
HNSCC16_P12_HNSCC16_P12_G10,13.572460,1.146078,Immune cells,CD8Tex,CD8Tex,T cell,3,Primary,T cell,HNSCC16,Tumor,63,Female,Primary,I
HNSCC16_P14_HNSCC16_P14_H10,12.089951,-0.791267,Immune cells,CD8Tex,CD8Tex,T cell,3,Primary,T cell,HNSCC16,Tumor,63,Female,Primary,I


In [24]:
res = cellmesh.run_cell_mesh(
        adata,
        # reaction_table=reaction_table,
        enzyme_metabolite=enzyme,
        metabolite_sensor=sensor,
        use_new_availability=True,
        cell_type_key="celltype",
        min_cells_per_group=1,
        allow_self=False,
        lower=0,
        upper=100
    )

In [25]:
res

CellMeshResult(events=             sender     receiver       metabolite      hmdb_id sensor_gene  \
0       Fibroblasts   Mono/Macro       Calcitriol  HMDB0001903         VDR   
1       Fibroblasts         CD8T       Calcitriol  HMDB0001903         VDR   
2       Fibroblasts    Malignant      Cholesterol  HMDB0000067        RORC   
3          CD4Tconv    Malignant      Cholesterol  HMDB0000067        RORC   
4       Fibroblasts       CD8Tex       Calcitriol  HMDB0001903         VDR   
..              ...          ...              ...          ...         ...   
985       Malignant       Plasma  Androstenedione  HMDB0000053          AR   
986  Myofibroblasts       Plasma  Androstenedione  HMDB0000053          AR   
987     Fibroblasts       Plasma  Androstenedione  HMDB0000053          AR   
988            CD8T       Plasma  Androstenedione  HMDB0000053          AR   
989        CD4Tconv  Endothelial  Androstenedione  HMDB0000053          AR   

          sensor_type  sender_score  rece